In [7]:
from collections import Counter
from pathlib import Path

from rdflib import Graph, Literal, Namespace, URIRef
from rdflib.namespace import OWL, RDF, RDFS, SKOS

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"

wittgensteiniana_path = DATA_DIR / "wittgensteiniana_final.ttl"
wab_path = DATA_DIR / "wab_tractatus_subgraph.ttl"

wittgensteiniana = Graph()
wittgensteiniana.parse(str(wittgensteiniana_path), format="turtle")

wab = Graph()
wab.parse(str(wab_path), format="turtle")

print("Wittgensteiniana loaded:", len(wittgensteiniana))
print("WAB loaded:", len(wab))

Wittgensteiniana loaded: 15567
WAB loaded: 27225


In [6]:
graphs = {
    "Wittgensteiniana": wittgensteiniana,
    "WAB": wab
}

for name, graph in graphs.items():
    print("\n")
    print(name)

    types = Counter(
        str(obj)
        for _, _, obj in graph.triples((None, RDF.type, None))
    )

    print("\nRDF types:")
    for rdf_type, count in types.most_common():
        print(f"{count:6} | {rdf_type}")

    print("\nTotal triples:", len(graph))



Wittgensteiniana

RDF types:
  1578 | http://example.org/wittgensteiniana/TextFragment
  1578 | https://schema.org/CreativeWork
   526 | http://example.org/wittgensteiniana/Thesis
   526 | https://schema.org/Chapter
   182 | https://schema.org/DefinedTerm
     8 | http://www.w3.org/2002/07/owl#Class
     7 | http://www.w3.org/2002/07/owl#ObjectProperty
     6 | http://www.w3.org/2002/07/owl#DatatypeProperty
     4 | http://example.org/wittgensteiniana/Author
     3 | https://schema.org/Book
     1 | https://schema.org/DefinedTermSet

Total triples: 15567


WAB

RDF types:
  1619 | http://purl.org/wittgensteinsource/ont/TS
  1616 | http://purl.org/wittgensteinsource/ont/Bemerkung
   531 | http://purl.org/wittgensteinsource/ont/Part
    21 | http://purl.org/wittgensteinsource/ont/scholar/0.1/Person
    13 | http://purl.org/wittgensteinsource/ont/WittgensteinExternalSource
    10 | http://www.w3.org/2002/07/owl#ObjectProperty
     6 | http://purl.org/wittgensteinsource/ont/scholar/0.1/T

In [10]:
SCHEMA = Namespace("https://schema.org/")
WAB_NS = Namespace("http://purl.org/wittgensteinsource/ont/")

label_properties = {
    "rdfs:label": RDFS.label,
    "skos:prefLabel": SKOS.prefLabel,
    "skos:altLabel": SKOS.altLabel,
    "schema:name": SCHEMA.name,
}

for name, graph in graphs.items():
    print("\n")
    print(name)

    for property_name, property_uri in label_properties.items():
        values = list(graph.triples((None, property_uri, None)))
        print(f"{property_name}: {len(values)}")




Wittgensteiniana
rdfs:label: 4
skos:prefLabel: 0
skos:altLabel: 0
schema:name: 712


WAB
rdfs:label: 2210
skos:prefLabel: 0
skos:altLabel: 0
schema:name: 0


In [11]:
for name, graph in graphs.items():
    print("\n")
    print(name)

    predicates = Counter(
        str(predicate)
        for _, predicate, _ in graph
        if predicate != RDF.type
    )

    print("\nMost frequent predicates:")

    for predicate, count in predicates.most_common(30):
        print(f"{count:6} | {predicate}")

    print("\nObject property declarations:")

    for prop in sorted(
        graph.subjects(RDF.type, OWL.ObjectProperty),
        key=str
    ):
        domains = list(graph.objects(prop, RDFS.domain))
        ranges = list(graph.objects(prop, RDFS.range))

        print(f"\nProperty: {prop}")
        print("Domain:", domains if domains else "Not specified")
        print("Range:", ranges if ranges else "Not specified")



Wittgensteiniana

Most frequent predicates:
  2104 | https://schema.org/isPartOf
  1581 | https://schema.org/inLanguage
  1578 | http://example.org/wittgensteiniana/hasTextFragment
  1578 | https://schema.org/exampleOfWork
  1578 | https://schema.org/text
  1095 | https://schema.org/mentions
   712 | https://schema.org/name
   526 | https://schema.org/position
   182 | https://schema.org/inDefinedTermSet
   177 | https://schema.org/description
    13 | http://www.w3.org/2000/01/rdf-schema#range
    11 | http://www.w3.org/2000/01/rdf-schema#domain
     4 | http://example.org/wittgensteiniana/hasAuthor
     4 | http://www.w3.org/2000/01/rdf-schema#label
     3 | https://schema.org/datePublished
     2 | https://schema.org/translationOfWork

Object property declarations:

Property: http://example.org/wittgensteiniana/hasAuthor
Domain: [rdflib.term.URIRef('https://schema.org/Book')]
Range: [rdflib.term.URIRef('http://example.org/wittgensteiniana/Author')]

Property: http://example.org/wi

In [16]:
from rdflib.namespace import RDF, RDFS

def get_resource_labels(graph, resource):
    labels = []

    for predicate, obj in graph.predicate_objects(resource):
        predicate_name = str(predicate)

        if predicate_name in [
            "http://www.w3.org/2000/01/rdf-schema#label",
            "https://schema.org/name"
        ]:
            labels.append(str(obj))

    return labels


types_to_inspect = {
    "Wittgensteiniana": [
        "http://example.org/wittgensteiniana/Thesis",
        "http://example.org/wittgensteiniana/TextFragment",
        "http://example.org/wittgensteiniana/Author",
        "https://schema.org/Book"
    ],
    "WAB": [
        "http://purl.org/wittgensteinsource/ont/Bemerkung",
        "http://purl.org/wittgensteinsource/ont/TS",
        "http://purl.org/wittgensteinsource/ont/Part",
        "http://purl.org/wittgensteinsource/ont/Person",
        "http://purl.org/wittgensteinsource/ont/Werk"
    ]
}

for graph_name, graph in graphs.items():
    print("\n")
    print(graph_name)

    for class_uri in types_to_inspect[graph_name]:
        class_uri = URIRef(class_uri)

        instances = sorted(
            graph.subjects(RDF.type, class_uri),
            key=str
        )

        print(f"\nClass: {class_uri.split('/')[-1]}")
        print(f"Number of instances: {len(instances)}")

        for instance in instances[:5]:
            labels = get_resource_labels(graph, instance)

            print("  URI:", instance)
            print("  Labels:", labels if labels else "No label")



Wittgensteiniana

Class: Thesis
Number of instances: 526
  URI: http://wittgensteiniana.michelepasin.org/id/unit/1000
  Labels: ['6.2341']
  URI: http://wittgensteiniana.michelepasin.org/id/unit/1001
  Labels: ['6.24']
  URI: http://wittgensteiniana.michelepasin.org/id/unit/1002
  Labels: ['6.241']
  URI: http://wittgensteiniana.michelepasin.org/id/unit/1003
  Labels: ['6.3']
  URI: http://wittgensteiniana.michelepasin.org/id/unit/1004
  Labels: ['6.31']

Class: TextFragment
Number of instances: 1578
  URI: http://wittgensteiniana.michelepasin.org/id/fragment/1000-german
  Labels: No label
  URI: http://wittgensteiniana.michelepasin.org/id/fragment/1000-ogden
  Labels: No label
  URI: http://wittgensteiniana.michelepasin.org/id/fragment/1000-pears
  Labels: No label
  URI: http://wittgensteiniana.michelepasin.org/id/fragment/1001-german
  Labels: No label
  URI: http://wittgensteiniana.michelepasin.org/id/fragment/1001-ogden
  Labels: No label

Class: Author
Number of instances: 4
  

In [14]:
print("Wittgensteiniana triples:", len(wittgensteiniana))
print("WAB triples:", len(wab))

print("\nAll classes containing 'Person' in WAB:")

person_classes = sorted(
    {
        obj
        for _, _, obj in wab.triples((None, RDF.type, None))
        if "Person" in str(obj)
    },
    key=str
)

for cls in person_classes:
    instances = list(wab.subjects(RDF.type, cls))
    print(f"\nClass URI: {cls}")
    print(f"Instances: {len(instances)}")

    for person in instances[:25]:
        print(" ", person)
        

Wittgensteiniana triples: 15567
WAB triples: 27225

All classes containing 'Person' in WAB:

Class URI: http://purl.org/wittgensteinsource/ont/scholar/0.1/Person
Instances: 21
  http://purl.org/wittgensteinsource/ont/instances/C%C3%A4sar__Gaius_Julius
  http://purl.org/wittgensteinsource/ont/instances/Darwin__Charles
  http://purl.org/wittgensteinsource/ont/instances/Engelmann__Paul
  http://purl.org/wittgensteinsource/ont/instances/Mauthner__Fritz
  http://purl.org/wittgensteinsource/ont/instances/Newton__Isaac
  http://purl.org/wittgensteinsource/ont/instances/Pinsent__David_Hume
  http://purl.org/wittgensteinsource/ont/instances/Sokrates
  http://purl.org/wittgensteinsource/ont/instances/Wanicek__Aloisia
  http://purl.org/wittgensteinsource/ont/instances/Ernst__Paul
  http://purl.org/wittgensteinsource/ont/instances/Grimm__Jacob%3B_Grimm__Wilhelm
  http://purl.org/wittgensteinsource/ont/instances/K%C3%BCrnberger__Ferdinand
  http://purl.org/wittgensteinsource/ont/instances/Kant__Imm

In [15]:
WAB_HAS_AUTHOR = URIRef(
    "http://purl.org/wittgensteinsource/ont/scholar/0.1/hasAuthor"
)

WAB_PERSON = URIRef(
    "http://purl.org/wittgensteinsource/ont/scholar/0.1/Person"
)

wab_persons = sorted(
    wab.subjects(RDF.type, WAB_PERSON),
    key=str
)

print("WAB persons:", len(wab_persons))

for person in wab_persons:
    labels = list(wab.objects(person, RDFS.label))

    author_of = list(wab.subjects(WAB_HAS_AUTHOR, person))

    print("\nPerson:", labels if labels else person)
    print("URI:", person)
    print("Linked as author:", len(author_of))

    for resource in author_of[:5]:
        resource_labels = list(wab.objects(resource, RDFS.label))
        print("  Author of:", resource_labels if resource_labels else resource)

WAB persons: 21

Person: [rdflib.term.Literal('Cäsar, Gaius Julius')]
URI: http://purl.org/wittgensteinsource/ont/instances/C%C3%A4sar__Gaius_Julius
Linked as author: 0

Person: [rdflib.term.Literal('Darwin, Charles')]
URI: http://purl.org/wittgensteinsource/ont/instances/Darwin__Charles
Linked as author: 0

Person: [rdflib.term.Literal('Engelmann, Paul')]
URI: http://purl.org/wittgensteinsource/ont/instances/Engelmann__Paul
Linked as author: 0

Person: [rdflib.term.Literal('Ernst, Paul')]
URI: http://purl.org/wittgensteinsource/ont/instances/Ernst__Paul
Linked as author: 0

Person: [rdflib.term.Literal('Frege, Gottlob')]
URI: http://purl.org/wittgensteinsource/ont/instances/Frege__Gottlob
Linked as author: 0

Person: [rdflib.term.Literal('Grimm, Jacob; Grimm, Wilhelm')]
URI: http://purl.org/wittgensteinsource/ont/instances/Grimm__Jacob%3B_Grimm__Wilhelm
Linked as author: 0

Person: [rdflib.term.Literal('Hertz, Heinrich')]
URI: http://purl.org/wittgensteinsource/ont/instances/Hertz__He

In [18]:
for person in wab_persons:
    label = next(wab.objects(person, RDFS.label), person)

    print("\n")
    print("Person:", label)

    print("Outgoing relations:")
    for predicate, obj in wab.predicate_objects(person):
        print(f"  {predicate.split('/')[-1]} -> {obj}")

    print("Incoming relations:")
    for subject, predicate in wab.subject_predicates(person):
        print(f"  {predicate.split('/')[-1]} <- {subject}")



Person: Cäsar, Gaius Julius
Outgoing relations:
  22-rdf-syntax-ns#type -> http://purl.org/wittgensteinsource/ont/scholar/0.1/Person
  rdf-schema#label -> Cäsar, Gaius Julius
  isReferredToIn -> http://wittgensteinsource.org/Ts-202%2C24r%5B8%5Det25r%5B1%5D_d
  isReferredToIn -> http://wittgensteinsource.org/Ts-203%2C27r%5B7%5D_d
  isReferredToIn -> http://wittgensteinsource.org/Ts-204%2C24r%5B8%5Det25r%5B1%5D_d
Incoming relations:


Person: Darwin, Charles
Outgoing relations:
  22-rdf-syntax-ns#type -> http://purl.org/wittgensteinsource/ont/scholar/0.1/Person
  rdf-schema#label -> Darwin, Charles
  isReferredToIn -> http://wittgensteinsource.org/Ts-202%2C15av%5B1%5D-relocated_d
  isReferredToIn -> http://wittgensteinsource.org/Ts-202%2C15r%5B7%5Det15av%5B1%5Det16r%5B1%5D_d
  isReferredToIn -> http://wittgensteinsource.org/Ts-204%2C16r%5B3%5D_d
Incoming relations:


Person: Engelmann, Paul
Outgoing relations:
  22-rdf-syntax-ns#type -> http://purl.org/wittgensteinsource/ont/scholar/0.